In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels
import yfinance as yf
from scipy import stats

plt.style.use('ggplot')
pd.set_option('display.max_rows', 50)
"""pd.set_option('display.width', 400)
pd.set_option('display.max_columns', 26)"""

"pd.set_option('display.width', 400)\npd.set_option('display.max_columns', 26)"

In [2]:
de_tickers = ['FRE.DE', 'BEI.DE', 'MRK.DE', 'FME.DE', 'ADS.DE', 'HEN3.DE', 'DHL.DE', 'CON.DE', 'TKA.DE', 'BMW.DE']

initial_df = yf.download(de_tickers, start='2005-01-01', end='2013-01-01', auto_adjust=True)['Close']

df = pd.DataFrame([initial_df.iloc[row] / initial_df.iloc[row-1] for row in range(1, initial_df.shape[0])], index=initial_df.index[1:])
df = np.log(df) # логарифмические доходности
df.dropna(inplace=True, axis=0) # удаляем пропущенные занчения


[*********************100%***********************]  10 of 10 completed


In [3]:
years = sorted(df.index.year.unique())
asset_names = df.columns.tolist()
n_assets = df.shape[1]
n_obs = len(df)
n_years = 2012 - 2005 + 1
gamma0 = 0.15 # treshold
confidence_level = 0.95

M = n_assets * (n_assets - 1) // 2 # кол-во пар
alpha = 1 - confidence_level
c_quantile = stats.norm.ppf(1 - alpha / M) # квантиль нормального распределения для процедур Бонферрони

Таблица -1, 0, 1, где строки - пары акций, а столбццы - года.   
*   1 = значимое ребро  
*   -1 = значимое неребро   
*   0 = неопределённость   

In [4]:
def get_significant_edges(returns):
  results = {}

  for year in years:
    year_data = returns[returns.index.year == year]
    corr_mat = year_data.corr() # корреляционная матрица

    # матрица с 1, -1, 0 значениями, где столбцы и строки - активы
    result_mat = pd.DataFrame(0, index=asset_names, columns=asset_names)

    for i in range(n_assets):
      for j in range(i + 1, n_assets):
        asset_i = asset_names[i]
        asset_j = asset_names[j]

        r_ij = corr_mat.loc[asset_i, asset_j]

        T = np.sqrt(n_obs - 1) * (r_ij - gamma0) / np.sqrt(1 - r_ij**2) # статистика

        # ручные тесты, без использования спец библиотек
        if T > c_quantile: # значимое ребро
          result_mat.loc[asset_i, asset_j] = 1
          result_mat.loc[asset_j, asset_i] = 1
        elif T < -c_quantile: # значимое неребро
          result_mat.loc[asset_i, asset_j] = -1
          result_mat.loc[asset_j, asset_i] = -1
        # else: остаётся 0 (неопределённость)

    # Сохраняем только верхний треугольник для удобства
    upper_tri = result_mat.where(np.triu(np.ones(result_mat.shape), k=1).astype(bool))
    results[year] = upper_tri

  return results

Функции для вывода таблицы, где столбцы и строки - года, а значения в ячейках:    
*   1 - значимая динамика есть
*   0 - принимается гипотеза об отсутствии динамики   


In [5]:
def get_test_T_statistic(returns):
    corr_mat = returns.corr()

    # матрица, состоящая из статистик
    T_mat = pd.DataFrame(np.nan, index=asset_names, columns=asset_names)

    for i in range(n_assets):
        for j in range(i + 1, n_assets):
            r_ij = corr_mat.iloc[i, j]

            T = np.sqrt(n_obs - 1) * (r_ij - gamma0) / np.sqrt(1 - r_ij**2)
            T_mat.iloc[i, j] = T
            T_mat.iloc[j, i] = T

    return T_mat

In [6]:
def get_significant_sets(T_mat):
    Le = []  # значимые рёбра (T > c)
    Ln = []  # значимые неребра (T < -c)

    for i in range(T_mat.shape[0]):
        for j in range(i + 1, T_mat.shape[1]):
            T_val = T_mat.iloc[i, j]
            if not np.isnan(T_val):
                if T_val > c_quantile:
                    Le.append((T_mat.index[i], T_mat.columns[j]))
                elif T_val < -c_quantile:
                    Ln.append((T_mat.index[i], T_mat.columns[j]))

    return set(Le), set(Ln)

In [7]:
def two_sample_test(returns1, returns2):
    # статистики для обоих периодов
    T1 = get_test_T_statistic(returns1)
    T2 = get_test_T_statistic(returns2)

    # Получаем множества
    Le1, Ln1 = get_significant_sets(T1)
    Le2, Ln2 = get_significant_sets(T2)

    # Проверяем условие: (Ln1 ∩ Le2) ∪ (Ln2 ∩ Le1) != ∅
    if (Ln1 & Le2) or (Ln2 & Le1):
        return 1  # динамика есть
    else:
        return 0  # динамики нет

Вывод таблиц:

In [8]:
results_by_year = get_significant_edges(df)

# собираем в одну таблицу
pairs = []
for i in range(len(asset_names)):
  for j in range(i + 1, len(asset_names)):
    pairs.append(f"({asset_names[i]}, {asset_names[j]})")

table = pd.DataFrame(index=pairs)

for year in results_by_year:
  year_results = []
  for pair in pairs:
    pair = pair[1:-1]
    a, b = pair.split(', ')
    val = results_by_year[year].loc[a, b]
    year_results.append(val)
  table[year] = year_results

table = table.astype(int)
table.index.name = 'pairs'
table.to_csv("./data/data_for_graphs.csv", index_label='pairs')
table

,2005,2006,2007,2008,2009,2010,2011,2012
pairs,,,,,,,,
"(ADS.DE, BEI.DE)",0,1,1,1,1,1,1,1
"(ADS.DE, BMW.DE)",0,1,1,1,1,1,1,1
"(ADS.DE, CON.DE)",1,1,1,1,1,1,1,1
"(ADS.DE, DHL.DE)",1,1,1,1,1,1,1,1
"(ADS.DE, FME.DE)",1,1,1,1,0,1,1,1
"(ADS.DE, FRE.DE)",0,1,1,1,0,0,1,1
"(ADS.DE, HEN3.DE)",1,1,1,1,1,1,1,1
"(ADS.DE, MRK.DE)",1,1,0,1,0,0,1,1
"(ADS.DE, TKA.DE)",1,1,1,1,1,1,1,1


In [9]:
table2 = pd.DataFrame(0, index=years, columns=years)

for i, year1 in enumerate(years):
    for j, year2 in enumerate(years):
        if i < j:  # только верхний треугольник
            data1 = df[df.index.year == year1]
            data2 = df[df.index.year == year2]

            result = two_sample_test(data1, data2)
            table2.loc[year1, year2] = result
            table2.loc[year2, year1] = result  # симметрично

print("Динамика между годами:")
print(table2)

print("\nПоследовательные годы:")
for i in range(len(years) - 1):
    year1 = years[i]
    year2 = years[i + 1]
    result = table2.loc[year1, year2]
    print(f"{year1} - {year2}: {result}")

Динамика между годами:
      2005  2006  2007  2008  2009  2010  2011  2012
2005     0     1     1     1     1     1     1     1
2006     1     0     0     0     1     1     0     0
2007     1     0     0     1     1     1     1     1
2008     1     0     1     0     1     0     0     0
2009     1     1     1     1     0     1     1     1
2010     1     1     1     0     1     0     1     1
2011     1     0     1     0     1     1     0     0
2012     1     0     1     0     1     1     0     0

Последовательные годы:
2005 - 2006: 1
2006 - 2007: 0
2007 - 2008: 1
2008 - 2009: 1
2009 - 2010: 1
2010 - 2011: 1
2011 - 2012: 0


Изменим ключевые параметры и посмотрим, как изменятся таблицы:

In [10]:
for some_gamma in [0.1, 0.13, 0.15, 0.17]:
  for level in [0.9, 0.95, 0.99]:
    print(f"\n{'='*50}")
    print(f"gamma0 = {some_gamma} and confidence level = {level}")
    print('='*50)

    gamma0 = some_gamma
    confidence_level = level
    M = n_assets * (n_assets - 1) // 2
    alpha = 1 - confidence_level
    c_quantile = stats.norm.ppf(1 - alpha / M)

    table2 = pd.DataFrame(0, index=years, columns=years)

    for i, year1 in enumerate(years):
        for j, year2 in enumerate(years):
            if i < j:
                data1 = df[df.index.year == year1]
                data2 = df[df.index.year == year2]
                result = two_sample_test(data1, data2)
                table2.loc[year1, year2] = result
                table2.loc[year2, year1] = result

    for i in range(len(years) - 1):
        year1 = years[i]
        year2 = years[i + 1]
        result = table2.loc[year1, year2]
        print(f"{year1} - {year2}: {result}")


gamma0 = 0.1 and confidence level = 0.9
2005 - 2006: 0
2006 - 2007: 0
2007 - 2008: 0
2008 - 2009: 1
2009 - 2010: 1
2010 - 2011: 0
2011 - 2012: 0

gamma0 = 0.1 and confidence level = 0.95
2005 - 2006: 0
2006 - 2007: 0
2007 - 2008: 0
2008 - 2009: 1
2009 - 2010: 1
2010 - 2011: 0
2011 - 2012: 0

gamma0 = 0.1 and confidence level = 0.99
2005 - 2006: 0
2006 - 2007: 0
2007 - 2008: 0
2008 - 2009: 1
2009 - 2010: 1
2010 - 2011: 0
2011 - 2012: 0

gamma0 = 0.13 and confidence level = 0.9
2005 - 2006: 1
2006 - 2007: 0
2007 - 2008: 0
2008 - 2009: 1
2009 - 2010: 1
2010 - 2011: 1
2011 - 2012: 0

gamma0 = 0.13 and confidence level = 0.95
2005 - 2006: 1
2006 - 2007: 0
2007 - 2008: 0
2008 - 2009: 1
2009 - 2010: 1
2010 - 2011: 1
2011 - 2012: 0

gamma0 = 0.13 and confidence level = 0.99
2005 - 2006: 1
2006 - 2007: 0
2007 - 2008: 0
2008 - 2009: 1
2009 - 2010: 1
2010 - 2011: 0
2011 - 2012: 0

gamma0 = 0.15 and confidence level = 0.9
2005 - 2006: 1
2006 - 2007: 0
2007 - 2008: 1
2008 - 2009: 1
2009 - 2010: 1


In [11]:
for some_gamma in [0.1, 0.13, 0.15, 0.17]:
    print(f"\n{'='*50}")
    print(f"gamma0 = {some_gamma}")
    print('='*50)

    gamma0 = some_gamma
    confidence_level = 0.99
    M = n_assets * (n_assets - 1) // 2
    alpha = 1 - confidence_level
    c_quantile = stats.norm.ppf(1 - alpha / M)

    table2 = pd.DataFrame(0, index=years, columns=years)

    for i, year1 in enumerate(years):
      for j, year2 in enumerate(years):
        if i < j:
            data1 = df[df.index.year == year1]
            data2 = df[df.index.year == year2]

            result = two_sample_test(data1, data2)
            table2.loc[year1, year2] = result
            table2.loc[year2, year1] = result
    print(table2)


gamma0 = 0.1
      2005  2006  2007  2008  2009  2010  2011  2012
2005     0     0     0     0     1     0     0     0
2006     0     0     0     0     1     0     0     0
2007     0     0     0     0     1     0     0     0
2008     0     0     0     0     1     0     0     0
2009     1     1     1     1     0     1     1     1
2010     0     0     0     0     1     0     0     0
2011     0     0     0     0     1     0     0     0
2012     0     0     0     0     1     0     0     0

gamma0 = 0.13
      2005  2006  2007  2008  2009  2010  2011  2012
2005     0     1     1     1     1     0     1     1
2006     1     0     0     0     1     0     0     0
2007     1     0     0     0     1     0     0     0
2008     1     0     0     0     1     0     0     0
2009     1     1     1     1     0     1     1     1
2010     0     0     0     0     1     0     0     0
2011     1     0     0     0     1     0     0     0
2012     1     0     0     0     1     0     0     0

gamma0 = 0.15
  

Выбор пал на **gamma0 = 0.15, confidence_level = 0.99**, так как   
*   2007–2008 пара имеет значение 0 - это правильно: финансовый кризис только начинался, но ВВП Германии упал всего на 0,1%, экономика ещё не вошла в острую фазу. Тест верно не фиксирует структурного сдвига.
*   Пара 2008-2009 дает значение 1 - именно в этот период ВВП рухнул на 2,5% (исторический рекорд), паника на рынках, корреляционные связи резко меняются.
*   Для 2011-2012 года значение 0 - после кризиса рынок стабилизировался, новых шоков нет.
*   Более строгий уровень 0,99 даёт меньше ложных срабатываний (например, при 0,95 пара 2007–2008 ошибочно показывает 1).

Сравним результаты с таблицами, полученными традиционным методом - простое сравнение коэффициентов корреляций с фиксированным порогом. Решение о динамике принимается, если графы рынка, соответствующие i−му и j−му годам, отличаются. Пусть порог тоже равен 0.15.

In [12]:
corr_list = []
for year in range(2005, 2013):
  curr_year_df = df[df.index.year == year]
  year_corr = curr_year_df.corr() # корреляционные матрицы
  year_corr_binary = (year_corr > gamma0).astype(int) # матрицы из 1 и 0
  corr_list.append(year_corr.where(np.triu(np.ones(year_corr.shape), k=1).astype(bool)))

In [13]:
traditional_table = pd.DataFrame(0, index=years, columns=years)

for i, year1 in enumerate(years):
  for j, year2 in enumerate(years):
    if i < j:
      res = 0 if corr_list[i].equals(corr_list[j]) else 1
      traditional_table.loc[year1, year2] = res
      traditional_table.loc[year2, year1] = res  # симметрично

traditional_table

,2005,2006,2007,2008,2009,2010,2011,2012
2005,0,1,1,1,1,1,1,1
2006,1,0,1,1,1,1,1,1
2007,1,1,0,1,1,1,1,1
2008,1,1,1,0,1,1,1,1
2009,1,1,1,1,0,1,1,1
2010,1,1,1,1,1,0,1,1
2011,1,1,1,1,1,1,0,1
2012,1,1,1,1,1,1,1,0


Традиционный метод показывает, что во все года была динамика - везде значение 1.   
Рассмотрим пары акций, ответветственные за динамику на графе рынка:


In [14]:
dynamics_pairs = pd.DataFrame(columns=['Пара акций', 'Количество 1', 'Количество -1'])

for pair in table.index:
    values = table.loc[pair]

    count_1 = (values == 1).sum()
    count_minus_1 = (values == -1).sum()

    # Добавляем в результат
    dynamics_pairs = pd.concat([dynamics_pairs, pd.DataFrame({
        'Пара акций': [pair],
        'Количество 1': [count_1],
        'Количество -1': [count_minus_1]
    })], ignore_index=True)

dynamics_pairs = dynamics_pairs.sort_values('Количество 1', ascending=False).reset_index(drop=True)
dynamics_pairs[(dynamics_pairs['Количество -1'] != 0) & (dynamics_pairs['Количество 1'] != 0)]

,Пара акций,Количество 1,Количество -1
32,"(CON.DE, FME.DE)",5,2
34,"(BEI.DE, MRK.DE)",5,1
36,"(CON.DE, FRE.DE)",5,1
39,"(BMW.DE, FME.DE)",5,1
40,"(DHL.DE, FME.DE)",5,1
42,"(DHL.DE, FRE.DE)",4,1
43,"(FME.DE, TKA.DE)",3,1
44,"(DHL.DE, MRK.DE)",3,1
